# generator.ipynb

# Assignment 4
### CSC 537: Deep Learning

**Author:** Xander Palermo — ajp2s@missouristate.edu <br>
**Professor:** Mukulika Ghosh <br>
**Date:** 27 April 2026

-----

### Imports

Libraries

In [123]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
import requests
import pickle


Custom dataset class

In [124]:
class ShakespeareDataset(Dataset):
    def __init__(self, data, sequence_length):
        self.data = data
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input = self.data[idx : idx + self.sequence_length]
        target = self.data[idx + 1 : idx + self.sequence_length + 1]
        return input, target

Global Variables

In [125]:
SEQUENCE_LENGTH = 50
RAW_TEXT_PATH = "dataset/tiny_shakespeare/tiny_shakespeare.txt"
DUMP_PATH = "dataset/shakespeare.pkl"

BATCH_SIZE = 256

## Generate Data

#### Obtain dataset

In [126]:
data = None

if not os.path.exists(RAW_TEXT_PATH):      # Request raw dataset and save to disk
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    data = response.text

    with open(RAW_TEXT_PATH, "w") as f:
        f.write(data)

else:                                       # Cache raw dataset file
    with open(RAW_TEXT_PATH, "r") as f:
        data = f.read()

data = str(data)

#### Preprocess

In [127]:
data = data.lower()

In [128]:
char = sorted(set(data))

char2idx = {ch: i for i, ch in enumerate(char)}
idx2char = {i : ch for i, ch in enumerate(char)}

In [129]:
data = [char2idx[char] for char in data]
data = torch.tensor(data,dtype=torch.long)

#### Create DataLoader & Save

In [130]:
dataset = ShakespeareDataset(data, SEQUENCE_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [131]:
with open(DUMP_PATH, "wb") as f:
    pickle.dump(dataloader, f)